<h1># Import Library</h1>

In [31]:
import numpy as np 
import pandas as pd 

<h1>Step 1: Load the Dataset</h1>

In [32]:
orders = pd.read_csv("../01_Raw/maple_orders_2025.csv")
products = pd.read_csv("../01_Raw/maple_products.csv")

<h1># work on a copy, keep the original safe</h1>

In [33]:
df = orders.copy()         
print("Starting rows:", len(df))

Starting rows: 4034


<h1>Step 2: Make column names simple</h1>

In [34]:
df.columns = df.columns.str.lower().str.replace("%", "pct")
df.columns

Index(['order_id', 'order_date', 'customer_id', 'product_id', 'quantity',
       'unit_price', 'discount_pct', 'sales_channel', 'region',
       'payment_method', 'order_status', 'shipping_cost', 'customer_rating'],
      dtype='str')

<h1>Step 3: Remove TEST orders</h1>

In [35]:
print(df[df["customer_id"] == "TEST"].shape[0])   # check
df = df[df["customer_id"] != "TEST"]               # fix
print("Rows now:", len(df))

9
Rows now: 4025


<h1>Step 4: Clean Customer ID</h1>

In [36]:
df["customer_id"] = df["customer_id"].str.strip().str.upper()

<h1>Step 5: Fix Region</h1>

In [37]:
print(df["region"].value_counts(dropna=False))    # check first

region
Northeast     993
West          943
Southeast     726
Midwest       631
Southwest     399
NaN            80
South-East     53
West           48
Mid West       47
NE             32
west           29
North East     24
northeast      20
Name: count, dtype: int64


In [38]:
region_map = {
    "North East": "Northeast",
    "northeast": "Northeast",
    "NE": "Northeast",
    "South-East": "Southeast",
    "Mid West": "Midwest",
    "west": "West"
}
df["region"] = df["region"].str.strip()            # removes "West " space problem
df["region"] = df["region"].replace(region_map)
df["region"] = df["region"].fillna("Unknown")      # Sarah's decision

print(df["region"].value_counts())                 # check again: only 6 values now

region
Northeast    1069
West         1020
Southeast     779
Midwest       678
Southwest     399
Unknown        80
Name: count, dtype: int64


<h1>Step 6: Fix Sales Channel</h1>

In [39]:
print(df["sales_channel"].value_counts())

sales_channel
Website      1730
Etsy          818
Instagram     712
Amazon        608
website        61
Web            51
instagram      23
IG             22
Name: count, dtype: int64


In [40]:
sales_map = {
    "website" : "Website" , 
    "instagram" : "Instagram" , 
    "IG" : "Instagram" ,
    "Web" : "Website"
}

df['sales_channel'] = df['sales_channel'].str.strip()
df['sales_channel'] = df['sales_channel'].replace(sales_map)

print(df['sales_channel'].value_counts())

sales_channel
Website      1842
Etsy          818
Instagram     757
Amazon        608
Name: count, dtype: int64


<h1>Step 7: Fix Payment Method</h1>

In [41]:
df["payment_method"] = df["payment_method"].fillna("Unknown")

In [42]:
df["payment_method"].sample(15)

3236      Apple Pay
3860         PayPal
755       Apple Pay
383       Apple Pay
1350         PayPal
2946        Unknown
1665         PayPal
1145         PayPal
1809    Credit Card
228       Apple Pay
3457    Credit Card
2170      Gift Card
2821         PayPal
3257         PayPal
3438         PayPal
Name: payment_method, dtype: str

<h1>Step 8: Fix dates</h1>

In [43]:
print(df["order_date"].sample(15))                 # see the different formats

3691    12/10/2025
1760    2025-06-03
1953    07/30/2025
1859    11/09/2025
3021    2025-04-07
2173    2025-12-26
665     2025-11-17
3010    2025-10-02
3378    2025-11-08
849     2025-10-21
2101    09/03/2025
2029    2025-08-26
3996    2025-07-07
3840    2025-10-15
4025    04/09/2025
Name: order_date, dtype: str


In [44]:
df["order_date"] = pd.to_datetime(df["order_date"], format="mixed")

In [45]:
print(df["order_date"].isnull().sum())             # should be 0

0


In [46]:
print(df["order_date"].min(), df["order_date"].max())  # should be Jan–Dec 2025

2025-01-01 00:00:00 2025-12-31 00:00:00


<h1>Step 9: Fix Unit Price (text → number)</h1>

In [47]:
df["unit_price"] = df["unit_price"].str.replace("$", "", regex=False)
df["unit_price"] = df["unit_price"].astype(float)

In [48]:
print(df["unit_price"].dtype)  

float64


<h1>Step 10: Fix Quantity</h1>

In [49]:
df.loc[df["quantity"] == 200, "quantity"] = 2 

In [50]:
print((df["quantity"] <= 0).sum())  

22


In [51]:
df = df[df["quantity"] > 0]   

<h1>Step 11: Fix Rating</h1>

In [52]:
df.loc[df["customer_rating"] > 5, "customer_rating"] = None

In [53]:
print(df["customer_rating"].max())

5.0


<h1>Step 12: Remove duplicates</h1>

In [54]:
print("Exact duplicates:", df.duplicated().sum())
df = df.drop_duplicates()

Exact duplicates: 130


In [55]:
print("Repeated Order IDs:", df["order_id"].duplicated().sum())
df = df.drop_duplicates(subset="order_id")

Repeated Order IDs: 5


<h1>Step 13: Clean the Products file</h1>

In [56]:
products.columns = products.columns.str.lower()
print(products["category"].value_counts()) 

category
Bedding     4
Decor       4
Candles     3
Kitchen     3
Bath        3
candles     1
Kitchen     1
Name: count, dtype: int64


In [57]:
products["category"] = products["category"].str.strip().str.title()

In [58]:
products["unit_cost"] = products["unit_cost"].fillna(27.00) 

In [59]:
print(products["category"].value_counts())  

category
Candles    4
Bedding    4
Kitchen    4
Decor      4
Bath       3
Name: count, dtype: int64


<h1>Step 14: Merge the two files</h1>

In [60]:
rows_before = len(df)

In [61]:
df = df.merge(products[["product_id", "product_name", "category", "unit_cost"]],
              on="product_id", how="left")

In [62]:
print(rows_before, len(df))                  
print(df["category"].isnull().sum()) 

3868 3868
0


<h1>Step 15: Add calculated columns</h1>

In [63]:
df["revenue"]    = df["quantity"] * df["unit_price"] * (1 - df["discount_pct"] / 100)
df["total_cost"] = df["quantity"] * df["unit_cost"]
df["profit"]     = df["revenue"] - df["total_cost"]

df["month"]      = df["order_date"].dt.month
df["month_name"] = df["order_date"].dt.strftime("%b")    # Jan, Feb...

df["revenue"] = df["revenue"].round(2)
df["profit"]  = df["profit"].round(2)

In [64]:
df

,order_id,order_date,customer_id,product_id,quantity,unit_price,discount_pct,sales_channel,region,payment_method,...,shipping_cost,customer_rating,product_name,category,unit_cost,revenue,total_cost,profit,month,month_name
0,52895,2025-11-10,C2365,P107,1,65.0,10,Etsy,West,PayPal,...,8.99,4.0,Waffle Throw Blanket,Bedding,32.59,58.5,32.59,25.91,11,Nov
1,53245,2025-12-01,C2223,P117,2,58.0,0,Website,West,PayPal,...,8.99,4.0,Turkish Towel Set,Bath,29.74,116.0,59.48,56.52,12,Dec
2,50498,2025-03-13,C1645,P111,1,55.0,0,Instagram,Midwest,Credit Card,...,0.00,3.0,Cast Iron Skillet,Kitchen,25.13,55.0,25.13,29.87,3,Mar
3,51374,2025-06-17,C1210,P114,1,42.0,0,Amazon,Midwest,Gift Card,...,8.99,5.0,Ceramic Vase,Decor,20.14,42.0,20.14,21.86,6,Jun
4,51346,2025-06-13,C2142,P104,1,45.0,10,Etsy,Midwest,Credit Card,...,12.50,5.0,Candle Gift Set,Candles,14.40,40.5,14.40,26.10,6,Jun
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3863,52791,2025-11-04,C2379,P113,2,48.0,0,Amazon,Midwest,Gift Card,...,0.00,5.0,Woven Wall Hanging,Decor,21.27,96.0,42.54,53.46,11,Nov
3864,53601,2025-12-19,C1907,P114,3,42.0,0,Website,Midwest,Apple Pay,...,0.00,5.0,Ceramic Vase,Decor,20.14,126.0,60.42,65.58,12,Dec
3865,50075,2025-01-12,C1802,P117,1,58.0,0,Instagram,Northeast,PayPal,...,12.50,NaN,Turkish Towel Set,Bath,29.74,58.0,29.74,28.26,1,Jan
3866,52225,2025-09-17,C2010,P103,1,16.0,5,Etsy,Southeast,Gift Card,...,8.99,5.0,Lavender Jar Candle,Candles,5.77,15.2,5.77,9.43,9,Sep


<h1>Step 16: Final check + export</h1>

In [68]:
print("Final rows:", len(df))

Final rows: 3868


In [69]:
print(df.isnull().sum())     # only customer_rating should have blanks

order_id             0
order_date           0
customer_id          0
product_id           0
quantity             0
unit_price           0
discount_pct         0
sales_channel        0
region               0
payment_method       0
order_status         0
shipping_cost        0
customer_rating    658
product_name         0
category             0
unit_cost            0
revenue              0
total_cost           0
profit               0
month                0
month_name           0
dtype: int64


In [70]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3868 entries, 0 to 3867
Data columns (total 21 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   order_id         3868 non-null   int64         
 1   order_date       3868 non-null   datetime64[us]
 2   customer_id      3868 non-null   str           
 3   product_id       3868 non-null   str           
 4   quantity         3868 non-null   int64         
 5   unit_price       3868 non-null   float64       
 6   discount_pct     3868 non-null   int64         
 7   sales_channel    3868 non-null   str           
 8   region           3868 non-null   str           
 9   payment_method   3868 non-null   str           
 10  order_status     3868 non-null   str           
 11  shipping_cost    3868 non-null   float64       
 12  customer_rating  3210 non-null   float64       
 13  product_name     3868 non-null   str           
 14  category         3868 non-null   str           
 15

In [71]:
df.shape

(3868, 21)

In [74]:
df.to_csv("../02_Cleaned/orders_cleaned.csv", index=False)

products.to_csv("../02_Cleaned/products_cleaned.csv", index=False)